# US ETF Z-Score Backtest

This notebook backtests a Yahoo Finance version of the staged allocation idea using:
- Aggressive assets: `QLD`, `SMH`
- Defensive assets: `TLT`, `GLD`
- Signal asset: `QLD`

The stage logic mirrors the current notebook concept:
- Z-score buckets at `0 / -1 / -2 / -3 / -4`
- Sticky stage strategy
- Bucket-following strategy
- Always-aggressive comparison
- Fixed stage-0 comparison


In [ ]:
from __future__ import annotations

import subprocess
import sys
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd

try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'yfinance'])
    import yfinance as yf

START_DATE = '2000-01-01'
END_DATE = date.today().isoformat()
CASH_RATIO = 0.03
SIGNAL_CODE = 'QLD'
AGGRESSIVE_CODES = ['QLD', 'SMH']
DEFENSIVE_CODES = ['TLT', 'GLD']
ALL_CODES = AGGRESSIVE_CODES + DEFENSIVE_CODES
ZSCORE_WINDOW = 60
STAGE_ATTACK_DEFENSE_RATIOS = {
    0: (0.5, 0.5),
    1: (0.6, 0.4),
    2: (0.7, 0.3),
    3: (0.8, 0.2),
    4: (0.9, 0.1),
    5: (1.0, 0.0),
}


In [ ]:
def compute_zscore(closes: pd.Series, window: int = ZSCORE_WINDOW) -> pd.Series:
    rolling_mean = closes.rolling(window=window, min_periods=window).mean()
    rolling_std = closes.rolling(window=window, min_periods=window).std(ddof=0)
    return ((closes - rolling_mean) / rolling_std.replace(0, pd.NA)).astype(float)

def map_zscore_to_stage(zscore: float) -> int:
    if zscore >= 0:
        return 0
    if zscore >= -1.0:
        return 1
    if zscore >= -2.0:
        return 2
    if zscore >= -3.0:
        return 3
    if zscore >= -4.0:
        return 4
    return 5

def resolve_applied_stage(previous_stage: int, bucket_stage: int) -> int:
    if bucket_stage == 0:
        return 0
    return max(previous_stage, bucket_stage)

def build_stage_target_weights(stage: int) -> dict[str, float]:
    attack_ratio, defense_ratio = STAGE_ATTACK_DEFENSE_RATIOS[stage]
    return {
        'QLD': attack_ratio * 0.6,
        'SMH': attack_ratio * 0.4,
        'TLT': defense_ratio * 0.6,
        'GLD': defense_ratio * 0.4,
    }


In [ ]:
raw = yf.download(
    ALL_CODES,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    actions=True,
    progress=False,
)
adj_close = raw['Adj Close'].dropna(how='all')
adj_close = adj_close.dropna()
daily_returns = adj_close.pct_change().fillna(0.0)
signal_closes = adj_close[SIGNAL_CODE]
signal_zscore = compute_zscore(signal_closes)
signal_frame = pd.DataFrame({'close': signal_closes, 'zscore': signal_zscore}).dropna()
aligned_zscore = signal_frame['zscore'].reindex(daily_returns.index).ffill()
signal_frame['week'] = signal_frame.index.to_series().apply(lambda d: (d.isocalendar().year, d.isocalendar().week))
rebalance_dates = signal_frame.groupby('week').apply(lambda frame: frame.index[0]).tolist()
rebalance_set = set(rebalance_dates)
signal_frame.tail()


In [ ]:
def run_backtest() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series, pd.Series]:
    sticky_value = 1.0
    bucket_value = 1.0
    aggressive_value = 1.0
    fixed_value = 1.0
    previous_stage = 0

    sticky_weights = pd.Series(build_stage_target_weights(0)) * (1 - CASH_RATIO)
    bucket_weights = pd.Series(build_stage_target_weights(0)) * (1 - CASH_RATIO)
    aggressive_weights = pd.Series({'QLD': 0.6, 'SMH': 0.4, 'TLT': 0.0, 'GLD': 0.0}) * (1 - CASH_RATIO)
    fixed_weights = pd.Series(build_stage_target_weights(0)) * (1 - CASH_RATIO)

    rows = []
    sticky_history = []
    bucket_history = []
    aggressive_history = []
    fixed_history = []

    for idx, current_date in enumerate(daily_returns.index):
        if idx > 0:
            signal_date = daily_returns.index[idx - 1]
            current_zscore = float(aligned_zscore.iloc[idx - 1])
            bucket_stage = map_zscore_to_stage(current_zscore)
            applied_stage = resolve_applied_stage(previous_stage, bucket_stage)
            stage_changed = applied_stage != previous_stage
            should_rebalance = (current_date in rebalance_set) or stage_changed

            bucket_weights = pd.Series(build_stage_target_weights(bucket_stage)) * (1 - CASH_RATIO)
            if should_rebalance:
                sticky_weights = pd.Series(build_stage_target_weights(applied_stage)) * (1 - CASH_RATIO)
                previous_stage = applied_stage
                rows.append({
                    'rebalance_date': current_date,
                    'signal_date': signal_date,
                    'zscore': current_zscore,
                    'bucket_stage': bucket_stage,
                    'applied_stage': applied_stage,
                    'stage_changed': stage_changed,
                })

        day_ret = daily_returns.loc[current_date]
        sticky_value *= 1 + float((sticky_weights * day_ret[sticky_weights.index]).sum())
        bucket_value *= 1 + float((bucket_weights * day_ret[bucket_weights.index]).sum())
        aggressive_value *= 1 + float((aggressive_weights * day_ret[aggressive_weights.index]).sum())
        fixed_value *= 1 + float((fixed_weights * day_ret[fixed_weights.index]).sum())

        sticky_history.append((current_date, sticky_value))
        bucket_history.append((current_date, bucket_value))
        aggressive_history.append((current_date, aggressive_value))
        fixed_history.append((current_date, fixed_value))

    stage_history = pd.DataFrame(rows)
    sticky_series = pd.Series(dict(sticky_history), name='sticky_zscore')
    bucket_series = pd.Series(dict(bucket_history), name='bucket_following')
    aggressive_series = pd.Series(dict(aggressive_history), name='always_aggressive')
    fixed_series = pd.Series(dict(fixed_history), name='fixed_stage0')
    return stage_history, sticky_series, bucket_series, aggressive_series, fixed_series

stage_history, sticky_series, bucket_series, aggressive_series, fixed_series = run_backtest()
stage_history.tail()


In [ ]:
def summarize(series: pd.Series) -> dict[str, float]:
    returns = series.pct_change().dropna()
    cumulative = float(series.iloc[-1] - 1.0)
    annualized = float((series.iloc[-1] ** (252 / max(len(returns), 1))) - 1) if len(returns) else 0.0
    drawdown = series / series.cummax() - 1
    mdd = float(drawdown.min()) if len(drawdown) else 0.0
    volatility = float(returns.std() * (252 ** 0.5)) if len(returns) else 0.0
    return {
        'cumulative_return': cumulative,
        'annualized_return': annualized,
        'max_drawdown': mdd,
        'annualized_volatility': volatility,
    }

summary = pd.DataFrame({
    'sticky_zscore': summarize(sticky_series),
    'bucket_following': summarize(bucket_series),
    'always_aggressive': summarize(aggressive_series),
    'fixed_stage0': summarize(fixed_series),
}).T
summary


In [ ]:
signal_plot = signal_frame.copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
signal_plot['close'].plot(ax=ax1, color='tab:blue', title=f'{SIGNAL_CODE} Price and Z-score')
ax1.set_ylabel('Adjusted Close')

signal_plot['zscore'].plot(ax=ax2, color='tab:orange', label='Z-score')
for level in [0, -1.0, -2.0, -3.0, -4.0]:
    ax2.axhline(level, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax2.set_ylabel('Z-score')
ax2.text(signal_plot.index[0], 0.3, 'Stage buckets: 0 / -1 / -2 / -3 / -4', fontsize=9, color='gray')
ax2.legend(loc='upper left')
plt.show()


In [ ]:
def annual_returns(series: pd.Series) -> pd.Series:
    yearly = series.resample('Y').last()
    returns = yearly.pct_change()
    returns.index = returns.index.year
    return returns

annual_return_table = pd.DataFrame({
    'sticky_zscore': annual_returns(sticky_series),
    'bucket_following': annual_returns(bucket_series),
    'always_aggressive': annual_returns(aggressive_series),
    'fixed_stage0': annual_returns(fixed_series),
})
annual_return_table.style.format('{:.2%}')


In [ ]:
plot_frame = pd.concat([sticky_series, bucket_series, aggressive_series, fixed_series], axis=1)
stage_plot = stage_history[['rebalance_date', 'applied_stage']].drop_duplicates(subset=['rebalance_date']).copy()
stage_plot['rebalance_date'] = pd.to_datetime(stage_plot['rebalance_date'])
stage_plot = stage_plot.set_index('rebalance_date').reindex(plot_frame.index).ffill().fillna(0)

fig, ax1 = plt.subplots(figsize=(14, 6))
plot_frame.plot(ax=ax1, title='Yahoo Finance US ETF Backtest with Applied Stage')
ax1.set_ylabel('Cumulative NAV')

ax2 = ax1.twinx()
ax2.step(plot_frame.index, stage_plot['applied_stage'], where='post', color='black', alpha=0.35, label='applied_stage')
ax2.set_ylabel('Applied Stage')
ax2.set_ylim(-0.2, 5.2)
ax2.set_yticks(range(6))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.show()

print('Note: higher applied_stage means more aggressive attack exposure.')
stage_history.tail(20)
